# 04 — Calibration Analysis

Computes calibration statistics on `candidate_level_discrete.csv`.

**Sections:**
1. Domain gap test: sports vs. political overpricing in the (0.02, 0.15] band.
2. Favorites band calibration (>50¢) with sub-band breakdown.
3. Fine sub-band calibration: nine price bands across the full range.

**Reads:** `data/candidate_level_discrete.csv`
**Writes:** console / table output only


In [1]:
import sys, os
sys.path.insert(0, os.path.dirname(os.path.abspath('__file__')))
from utils import (clopper_pearson_upper, RNG_SEED, BOOT_REPS,
                   FLOOR_MAX, LS_MAX, FAV_MIN)

import pandas as pd
import numpy as np
from scipy import stats as scipy_stats
import warnings
warnings.filterwarnings('ignore')

BASE_DIR = os.path.abspath('..')
DATA_DIR = os.path.join(BASE_DIR, 'data')

np.random.seed(RNG_SEED)

cand_discrete = pd.read_csv(os.path.join(DATA_DIR, 'candidate_level_discrete.csv'))
print(f'candidate_level_discrete : {len(cand_discrete):,} rows  |  '
      f'{cand_discrete["event"].nunique()} events')
print(f'Domains: {dict(cand_discrete["domain"].value_counts())}')


candidate_level_discrete : 1,579 rows  |  133 events
Domains: {'Sports': np.int64(941), 'Political': np.int64(638)}


In [2]:
# ═══════════════════════════════════════════════════════════════════════════
# DOMAIN GAP TEST — Above-floor longshot band (0.02 < price ≤ 0.15)
#
# sports_gap  = win_rate(Sports in band) − mean_implied(Sports in band)
# pol_gap     = win_rate(Political in band) − mean_implied(Political in band)
# diff        = sports_gap − pol_gap
#
# Bootstrap: resample events with replacement (event-clustered, 2000 reps).
# 95% CI on diff via percentile method; significant if CI excludes 0.
# Dataset: cand_discrete (primary, excludes 24 numeric-range events).
# ═══════════════════════════════════════════════════════════════════════════

np.random.seed(RNG_SEED)

print('=' * 72)
print('DOMAIN GAP TEST — Above-floor longshot band (0.02 < price ≤ 0.15)')
print(f'Dataset: cand_discrete  ({cand_discrete["event"].nunique()} discrete-entity events)')
print('=' * 72)
print()

_dg_band = cand_discrete[
    (cand_discrete['snapshot_price'] > FLOOR_MAX) &
    (cand_discrete['snapshot_price'] <= 0.15)
].copy()

_dg_sp = _dg_band[_dg_band['domain'] == 'Sports']
_dg_po = _dg_band[_dg_band['domain'] == 'Political']

print('Band composition:')
print(f'  Sports    : {len(_dg_sp):>5,} candidates,  '
      f'{_dg_sp["event"].nunique():>3} events,  {int(_dg_sp["won"].sum()):>3} winners')
print(f'  Political : {len(_dg_po):>5,} candidates,  '
      f'{_dg_po["event"].nunique():>3} events,  {int(_dg_po["won"].sum()):>3} winners')
print()


def _domain_gap(df):
    """win_rate − mean_implied for a sub-frame. Returns nan if empty."""
    if len(df) == 0:
        return float('nan')
    return float(df['won'].mean()) - float(df['snapshot_price'].mean())


_sp_gap_pt  = _domain_gap(_dg_sp)
_po_gap_pt  = _domain_gap(_dg_po)
_diff_pt    = _sp_gap_pt - _po_gap_pt

print('Point estimates:')
print(f'  sports_gap   = win_rate {_dg_sp["won"].mean():.4f} '
      f'− implied {_dg_sp["snapshot_price"].mean():.4f} = {_sp_gap_pt:+.4f}')
print(f'  pol_gap      = win_rate {_dg_po["won"].mean():.4f} '
      f'− implied {_dg_po["snapshot_price"].mean():.4f} = {_po_gap_pt:+.4f}')
print(f'  diff (S − P) = {_diff_pt:+.4f}')
print()

# Event-clustered block bootstrap: resample events (not candidates).
# In each rep, gap is computed on whichever domain-specific candidates
# happen to appear in the resampled event set.
_dg_ev_groups = {
    ev: g.reset_index(drop=True) for ev, g in _dg_band.groupby('event')
}
_dg_evs      = list(_dg_ev_groups.keys())
_dg_boot     = np.full(BOOT_REPS, np.nan)

for _r in range(BOOT_REPS):
    _samp  = np.random.choice(_dg_evs, size=len(_dg_evs), replace=True)
    _bdf   = pd.concat([_dg_ev_groups[_e] for _e in _samp], ignore_index=True)
    _b_sp  = _bdf[_bdf['domain'] == 'Sports']
    _b_po  = _bdf[_bdf['domain'] == 'Political']
    if len(_b_sp) == 0 or len(_b_po) == 0:
        continue   # domain wiped out in this rep — skip (degenerate)
    _dg_boot[_r] = _domain_gap(_b_sp) - _domain_gap(_b_po)

_dg_valid    = _dg_boot[~np.isnan(_dg_boot)]
_dg_n_valid  = len(_dg_valid)
_dg_ci_lo, _dg_ci_hi = np.percentile(_dg_valid, [2.5, 97.5])
_significant = 'YES' if (_dg_ci_lo > 0 or _dg_ci_hi < 0) else 'NO'

print(f'Event-clustered block bootstrap ({BOOT_REPS} reps, {_dg_n_valid} valid):')
print(f'  95% CI on diff (sports_gap − pol_gap): [{_dg_ci_lo:+.4f}, {_dg_ci_hi:+.4f}]')
print(f'  Degenerate reps skipped              : {BOOT_REPS - _dg_n_valid}')
print()
print(f'Domain difference significant at 5%: {_significant}')


DOMAIN GAP TEST — Above-floor longshot band (0.02 < price ≤ 0.15)
Dataset: cand_discrete  (133 discrete-entity events)

Band composition:
  Sports    :   208 candidates,   34 events,    4 winners
  Political :   161 candidates,   70 events,    6 winners

Point estimates:
  sports_gap   = win_rate 0.0192 − implied 0.0609 = -0.0416
  pol_gap      = win_rate 0.0373 − implied 0.0736 = -0.0363
  diff (S − P) = -0.0053



Event-clustered block bootstrap (2000 reps, 2000 valid):
  95% CI on diff (sports_gap − pol_gap): [-0.0395, +0.0271]
  Degenerate reps skipped              : 0

Domain difference significant at 5%: NO


In [3]:
from scipy import stats as scipy_stats

def _wilson_ci(k, n, alpha=0.05):
    if n == 0: return float('nan'), float('nan')
    p  = k / n
    z  = scipy_stats.norm.ppf(1 - alpha / 2)
    d  = 1 + z**2 / n
    c  = (p + z**2 / (2 * n)) / d
    m  = z * np.sqrt(p * (1 - p) / n + z**2 / (4 * n**2)) / d
    return c - m, c + m

# ═══════════════════════════════════════════════════════════════════════════
# FAVORITES BAND — Calibration Report
# snapshot_price > 0.50  |  cand_discrete (primary, excludes numeric-range)
# Bootstrap: same event-clustered block logic as _band_stats (defined above).
# ═══════════════════════════════════════════════════════════════════════════

np.random.seed(RNG_SEED)

FAV_LO = 0.50


def _fav_boot_ci(df, reps=BOOT_REPS):
    # Event-clustered bootstrap for win rate on an arbitrary sub-frame.
    n      = len(df)
    n_wins = int(df['won'].sum()) if n else 0
    if n == 0:
        return float('nan'), float('nan'), '-'
    if n_wins == 0:
        return 0.0, clopper_pearson_upper(0, n, alpha=0.05), 'CP'
    ev_g = {ev: g.reset_index(drop=True) for ev, g in df.groupby('event')}
    evs  = list(ev_g.keys())
    boot = []
    for _ in range(reps):
        samp = np.random.choice(evs, size=len(evs), replace=True)
        boot.append(
            float(pd.concat([ev_g[e] for e in samp], ignore_index=True)['won'].mean())
        )
    return float(np.percentile(boot, 2.5)), float(np.percentile(boot, 97.5)), 'bootstrap'


def _fav_stats(df, label):
    n      = len(df)
    n_ev   = int(df['event'].nunique()) if n else 0
    n_wins = int(df['won'].sum())       if n else 0
    imp    = float(df['snapshot_price'].mean()) if n else float('nan')
    wr     = float(df['won'].mean())            if n else float('nan')
    gap    = wr - imp                           if n else float('nan')
    lo, hi, meth = _fav_boot_ci(df)
    wlo, whi     = _wilson_ci(n_wins, n)        if n else (float('nan'), float('nan'))
    return dict(label=label, n=n, n_ev=n_ev, n_wins=n_wins,
                imp=imp, wr=wr, gap=gap,
                lo=lo, hi=hi, meth=meth, wlo=wlo, whi=whi)


# ── Sub-frames ────────────────────────────────────────────────────────────
fav     = cand_discrete[cand_discrete['snapshot_price'] >  FAV_LO].copy()
fav_mid = cand_discrete[(cand_discrete['snapshot_price'] >  0.50) &
                         (cand_discrete['snapshot_price'] <= 0.80)].copy()
fav_hi  = cand_discrete[cand_discrete['snapshot_price'] >  0.80].copy()
fav_sp  = fav[fav['domain'] == 'Sports'].copy()
fav_po  = fav[fav['domain'] == 'Political'].copy()

rows_main = [
    _fav_stats(fav,     'Overall  (> 0.50)     '),
    _fav_stats(fav_mid, '  Sub-band (0.50,0.80]'),
    _fav_stats(fav_hi,  '  Sub-band (0.80,1.00]'),
]
rows_dom = [
    _fav_stats(fav_sp, 'Sports    (> 0.50)'),
    _fav_stats(fav_po, 'Political (> 0.50)'),
]

# ── Formatting helpers ────────────────────────────────────────────────────
def _ci(lo, hi):
    return f'[{lo:.4f},{hi:.4f}]' if not np.isnan(lo) else '      (-)     '

W   = 118
SEP = '  ' + '-' * (W - 2)

# ── Print ─────────────────────────────────────────────────────────────────
print('=' * W)
print('FAVORITES BAND — Calibration Report')
print(f'Dataset: cand_discrete  ({cand_discrete["event"].nunique()} events)  |  '
      f'{BOOT_REPS} bootstrap reps  |  event-clustered CIs')
print('=' * W)

print()
print('PARTS 1 & 3 — Overall and sub-band calibration')
HDR = (f'  {"Band":<26}  {"N_cand":>6}  {"N_ev":>4}  {"N_win":>5}  '
       f'{"Implied":>8}  {"WinRate":>8}  {"Gap":>7}  '
       f'{"Clustered 95%CI":^16}  {"Wilson 95%CI":^16}  CI method')
print(HDR)
print(SEP)
for r in rows_main:
    print(f'  {r["label"]:<26}  {r["n"]:>6,}  {r["n_ev"]:>4}  {r["n_wins"]:>5}  '
          f'{r["imp"]:>8.4f}  {r["wr"]:>8.4f}  {r["gap"]:>+7.4f}  '
          f'{_ci(r["lo"], r["hi"]):^16}  {_ci(r["wlo"], r["whi"]):^16}  {r["meth"]}')

# ── Verdicts ──────────────────────────────────────────────────────────────
ov   = rows_main[0]
cal  = not np.isnan(ov['lo']) and ov['lo'] <= ov['imp'] <= ov['hi']
undp = not np.isnan(ov['lo']) and ov['lo'] > ov['imp']

print()
print('Verdicts (Overall > 0.50):')
print(f'  Consistent with calibration?               '
      f'{"YES" if cal  else "NO "}  '
      f'(implied {ov["imp"]:.4f} {"inside" if cal else "outside"} '
      f'clustered CI {_ci(ov["lo"], ov["hi"])})')
print(f'  Any sign of classic favorite underpricing?  '
      f'{"YES" if undp else "NO "}  '
      f'(CI lower bound {ov["lo"]:.4f} {">" if undp else "<="} implied {ov["imp"]:.4f})')

# ── Domain descriptive ────────────────────────────────────────────────────
print()
print('=' * W)
print('PART 4 — DESCRIPTIVE: by domain (> 0.50 only)')
print('Small per-domain samples; CIs overlap heavily. NOT a domain-difference significance claim.')
HDR2 = (f'  {"Domain":<22}  {"N_cand":>6}  {"N_ev":>4}  {"N_win":>5}  '
        f'{"Implied":>8}  {"WinRate":>8}  {"Gap":>7}  '
        f'{"Clustered 95%CI":^16}  CI method')
print(HDR2)
print('  ' + '-' * 90)
for r in rows_dom:
    print(f'  {r["label"]:<22}  {r["n"]:>6,}  {r["n_ev"]:>4}  {r["n_wins"]:>5}  '
          f'{r["imp"]:>8.4f}  {r["wr"]:>8.4f}  {r["gap"]:>+7.4f}  '
          f'{_ci(r["lo"], r["hi"]):^16}  {r["meth"]}')
print()
print('  Interpret domain gaps as descriptive only.')


FAVORITES BAND — Calibration Report
Dataset: cand_discrete  (133 events)  |  2000 bootstrap reps  |  event-clustered CIs

PARTS 1 & 3 — Overall and sub-band calibration
  Band                        N_cand  N_ev  N_win   Implied   WinRate      Gap  Clustered 95%CI     Wilson 95%CI    CI method
  --------------------------------------------------------------------------------------------------------------------
  Overall  (> 0.50)               99    96     83    0.7908    0.8384  +0.0475  [0.7670,0.9072]   [0.7535,0.8980]   bootstrap
    Sub-band (0.50,0.80]          45    44     31    0.6816    0.6889  +0.0073  [0.5556,0.8182]   [0.5434,0.8047]   bootstrap
    Sub-band (0.80,1.00]          54    54     52    0.8819    0.9630  +0.0810  [0.9074,1.0000]   [0.8746,0.9898]   bootstrap

Verdicts (Overall > 0.50):
  Consistent with calibration?               YES  (implied 0.7908 inside clustered CI [0.7670,0.9072])
  Any sign of classic favorite underpricing?  NO   (CI lower bound 0.7670 <= 

## Calibration Regression

Linear probability models (LPM) with cluster-robust SEs (clustered by event)
on the **above-floor** sample (`snapshot_price > FLOOR_MAX`).  Dropping the
floored ≤2¢ tier removes the tick-floor artifact and lets the regression span
the full pricing range — longshots, mid, and favorites together.

* **Model 1** — Calibration headline: `won ~ snapshot_price`
* **Model 2** — Domain interaction: `won ~ snapshot_price * C(domain)`

Perfect calibration ↔ α = 0, β = 1 simultaneously.

In [4]:
import sys, os
import subprocess as _sp

# statsmodels — install quietly if missing
try:
    import statsmodels.formula.api as smf
    import statsmodels.api as sm
except ImportError:
    _sp.run([sys.executable, '-m', 'pip', 'install', '--quiet', 'statsmodels'],
            check=True)
    import statsmodels.formula.api as smf
    import statsmodels.api as sm

import pandas as pd
import numpy as np
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')

sys.path.insert(0, os.path.dirname(os.path.abspath('__file__')))
from utils import FLOOR_MAX, RNG_SEED, FL, FT, FA

BASE_DIR = os.path.abspath('..')
DATA_DIR = os.path.join(BASE_DIR, 'data')
OUT_DIR  = os.path.join(BASE_DIR, 'output')

# ── Load and filter ────────────────────────────────────────────────────────────
cand = pd.read_csv(os.path.join(DATA_DIR, 'candidate_level_discrete.csv'))

ab = cand[cand['snapshot_price'] > FLOOR_MAX].copy().reset_index(drop=True)

N       = len(ab)
N_clust = ab['event'].nunique()
EXPECTED_N = 587   # 369 (longshots) + 119 (mid) + 99 (favorites)

print(f'Above-floor sample  N        = {N:,}  (expected ~{EXPECTED_N})')
print(f'Event clusters               = {N_clust}')
if abs(N - EXPECTED_N) > 10:
    print(f'WARNING: N={N} differs from expected {EXPECTED_N} by more than 10 — check filter.')
else:
    print('Sanity check OK.')
print()
print('Domain breakdown:')
for dom, grp in ab.groupby('domain'):
    print(f'  {dom:<12}: {len(grp):,} candidates  {grp["event"].nunique()} events')

Above-floor sample  N        = 587  (expected ~587)
Event clusters               = 132
Sanity check OK.

Domain breakdown:
  Political   : 305 candidates  93 events
  Sports      : 282 candidates  39 events


In [5]:
# ══════════════════════════════════════════════════════════════════════════════
# MODEL 1 — Calibration headline:  won ~ snapshot_price
# ══════════════════════════════════════════════════════════════════════════════

np.random.seed(RNG_SEED)

m1 = smf.ols('won ~ snapshot_price', data=ab).fit(
    cov_type='cluster', cov_kwds={'groups': ab['event']}
)

alpha = m1.params['Intercept']
beta  = m1.params['snapshot_price']
se_a  = m1.bse['Intercept']
se_b  = m1.bse['snapshot_price']
ci_a  = m1.conf_int().loc['Intercept'].values
ci_b  = m1.conf_int().loc['snapshot_price'].values
p_a   = m1.pvalues['Intercept']
p_b   = m1.pvalues['snapshot_price']

# ── Single tests ──────────────────────────────────────────────────────────────
# β = 1  (slope = perfect calibration)
t_b1 = (beta - 1.0) / se_b
from scipy import stats as _sst
df_resid = N - 2
p_beta1  = float(2 * _sst.t.sf(abs(t_b1), df=df_resid))

# α = 0  (same as standard p-value)
p_alpha0 = float(p_a)

# ── Joint test: α = 0 AND β = 1 (Wald test, uses cluster-robust vcov) ─────────
joint = m1.wald_test('Intercept = 0, snapshot_price = 1', use_f=True)
p_joint = float(joint.pvalue)
f_joint = float(joint.statistic.flat[0])

print('=' * 72)
print('MODEL 1 — Calibration:  won ~ snapshot_price')
print(f'N = {N:,} candidates  |  {N_clust} event clusters  |  cluster-robust SEs')
print('=' * 72)
print()
print(f'  {"Parameter":<22}  {"Coef":>8}  {"Robust SE":>10}  '
      f'{"95% CI":^22}  {"p-value":>9}')
print('  ' + '-' * 76)
print(f'  {"Intercept  (α)":<22}  {alpha:>8.4f}  {se_a:>10.4f}  '
      f'[{ci_a[0]:>7.4f}, {ci_a[1]:>7.4f}]  {p_a:>9.4f}')
print(f'  {"snapshot_price (β)":<22}  {beta:>8.4f}  {se_b:>10.4f}  '
      f'[{ci_b[0]:>7.4f}, {ci_b[1]:>7.4f}]  {p_b:>9.4f}')
print('  ' + '-' * 76)
print(f'  {"β = 1 (H₀: slope = 1)":<22}  {"":>8}  {"":>10}  {"":^22}  '
      f'p = {p_beta1:.4f}')
print(f'  {"α = 0 (H₀: intercept=0)":<22}  {"":>8}  {"":>10}  {"":^22}  '
      f'p = {p_alpha0:.4f}')
print()
print(f'  JOINT NULL α=0, β=1  →  F = {f_joint:.3f},  p = {p_joint:.4f}')
print()

# Interpretation
print('─' * 72)
print('INTERPRETATION')
print('─' * 72)
print(f'  α = {alpha:.4f}  ({"< 0" if alpha < 0 else ">= 0"}),  '
      f'β = {beta:.4f}  ({"< 1" if beta < 1 else ">= 1"})')
if alpha < 0 and beta >= 1:
    print('  Pattern: α < 0, β ≥ 1 — classic favorite-longshot signature.')
    print('  The fitted line sits BELOW the 45° identity at the low (longshot) end')
    print('  and crosses above it at the high (favorite) end.')
elif alpha < 0 and beta < 1:
    print('  Pattern: α < 0, β < 1 — attenuation (regression-toward-mean).')
    print('  Longshots overpriced AND favorites overpriced; line is flatter than 45°.')
else:
    print('  Pattern does not match the classic FLB α<0, β≥1 signature.')
print()
print(f'  Joint calibration test (α=0, β=1): p = {p_joint:.4f}  '
      f'— {"reject" if p_joint < 0.05 else "cannot reject"} perfect calibration at 5%.')

MODEL 1 — Calibration:  won ~ snapshot_price
N = 587 candidates  |  132 event clusters  |  cluster-robust SEs

  Parameter                   Coef   Robust SE          95% CI            p-value
  ----------------------------------------------------------------------------
  Intercept  (α)           -0.0598      0.0088  [-0.0771, -0.0425]     0.0000
  snapshot_price (β)        1.1388      0.0381  [ 1.0642,  1.2135]     0.0000
  ----------------------------------------------------------------------------
  β = 1 (H₀: slope = 1)                                                 p = 0.0003
  α = 0 (H₀: intercept=0)                                                p = 0.0000

  JOINT NULL α=0, β=1  →  F = 26.494,  p = 0.0000

────────────────────────────────────────────────────────────────────────
INTERPRETATION
────────────────────────────────────────────────────────────────────────
  α = -0.0598  (< 0),  β = 1.1388  (>= 1)
  Pattern: α < 0, β ≥ 1 — classic favorite-longshot signature.
  The fi

In [6]:
# ══════════════════════════════════════════════════════════════════════════════
# MODEL 2 — Domain interaction:  won ~ snapshot_price * C(domain)
# ══════════════════════════════════════════════════════════════════════════════
# Reference level: Political (alphabetical first).
# Interaction term snapshot_price:C(domain)[T.Sports] tests whether the
# calibration slope differs between Sports and Political.

m2 = smf.ols('won ~ snapshot_price * C(domain)', data=ab).fit(
    cov_type='cluster', cov_kwds={'groups': ab['event']}
)

print('=' * 72)
print('MODEL 2 — Domain interaction:  won ~ snapshot_price * C(domain)')
print(f'N = {N:,}  |  {N_clust} clusters  |  reference domain = Political')
print('=' * 72)
print()

param_labels = {
    'Intercept':                          'Intercept (Political, α_P)',
    'snapshot_price':                     'snapshot_price (slope, Political)',
    'C(domain)[T.Sports]':                'Sports main effect (Δintercept)',
    'snapshot_price:C(domain)[T.Sports]': 'snapshot_price × Sports (Δslope)',
}

ci2 = m2.conf_int()
print(f'  {"Parameter":<38}  {"Coef":>8}  {"Robust SE":>10}  '
      f'{"95% CI":^22}  {"p-value":>9}')
print('  ' + '-' * 92)
for pname, label in param_labels.items():
    c   = m2.params[pname]
    se  = m2.bse[pname]
    cil = ci2.loc[pname, 0]
    cih = ci2.loc[pname, 1]
    pv  = m2.pvalues[pname]
    print(f'  {label:<38}  {c:>8.4f}  {se:>10.4f}  '
          f'[{cil:>7.4f}, {cih:>7.4f}]  {pv:>9.4f}')

print()
print('─' * 72)
print('INTERPRETATION — Domain interaction')
print('─' * 72)
slope_int_p = m2.pvalues.get('snapshot_price:C(domain)[T.Sports]', float('nan'))
main_eff_p  = m2.pvalues.get('C(domain)[T.Sports]', float('nan'))
slope_int_c = m2.params.get('snapshot_price:C(domain)[T.Sports]', float('nan'))

print(f'  Interaction (Δslope Sports−Political): coef={slope_int_c:.4f}  '
      f'p={slope_int_p:.4f}')
if slope_int_p >= 0.05:
    print('  → NOT significant: calibration slopes are statistically')
    print('    indistinguishable across domains at 5%.')
    print('    This formalises the "domains indistinguishable" finding.')
else:
    print(f'  → SIGNIFICANT (p={slope_int_p:.4f}): slopes differ by domain.')

print(f'  Domain main effect (Sports intercept shift): p={main_eff_p:.4f}')

MODEL 2 — Domain interaction:  won ~ snapshot_price * C(domain)
N = 587  |  132 clusters  |  reference domain = Political

  Parameter                                   Coef   Robust SE          95% CI            p-value
  --------------------------------------------------------------------------------------------
  Intercept (Political, α_P)               -0.0630      0.0154  [-0.0933, -0.0327]     0.0000
  snapshot_price (slope, Political)         1.1358      0.0486  [ 1.0405,  1.2311]     0.0000
  Sports main effect (Δintercept)           0.0043      0.0184  [-0.0318,  0.0404]     0.8151
  snapshot_price × Sports (Δslope)          0.0240      0.0913  [-0.1549,  0.2029]     0.7924

────────────────────────────────────────────────────────────────────────
INTERPRETATION — Domain interaction
────────────────────────────────────────────────────────────────────────
  Interaction (Δslope Sports−Political): coef=0.0240  p=0.7924
  → NOT significant: calibration slopes are statistically
    